# Part 2: Web 検索結果からの grounding を追加する

新人社員から「Zava の health benefits は業界水準に対して競争力があるか？」と聞かれました。Zava 社内の制度内容は分かっていても、後半に答えるには最新の業界ベンチマークが必要です。社内ドキュメントと Web の両方を検索できる knowledge base にうってつけの仕事です。

**🎯 ミッション**
- **Web IQ** を使って Web を検索できる **MCP knowledge source** を追加する
- 3 ソースの knowledge base（HR docs + health docs + Web）を構築する
- 片方のサブ回答が Zava 社内の health docs から、もう片方が Web ページから返るハイブリッドな質問を投げる


## 構成要素

Part 1 では knowledge base はビルド済みの検索インデックスのみをクエリしていました。ここでは 3 つ目のソース種別である **MCP Server Knowledge Source** を追加し、新しい Web IQ サービスに接続します。

- **MCP Server Knowledge Source**: 任意のリモート MCP エンドポイントに接続します。KB はクエリ時にツールのように呼び出します。
- **Web IQ**: MCP サーバーとして公開されている Microsoft の Web grounding サービスです。引用付きでランク付けされた Web 結果を返します。

## Step 1: 環境変数をセットアップする

Azure リソースの設定を読み込みます。プロジェクトフォルダーの `.env` ファイルには Azure AI Search のエンドポイント、Azure OpenAI の認証情報、モデル構成が *ほぼ* すべて入っています。

‼️ それに加えて `WEB_IQ_KEY` 変数が必要です。ラボの講師から、この値が記載されたファイルへのリンクが提供されます。現在の `.env` の末尾に追記してください。

```shell
WEB_IQ_KEY="some-value-here"
```

その後、下のセルを実行して環境変数を読み込みます。

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
WEB_IQ_KEY = os.environ["WEB_IQ_KEY"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")


## Step 2: 3 つの knowledge source を作成する

この knowledge base 用に、まず 3 つの knowledge source を作ります。最初のノートブックと同じ 2 つの index ベースのソースに加え、Web IQ MCP サーバー用の knowledge source を追加します。

* `healthdocs-knowledge-source`: `healthdocs` 検索インデックスを参照
* `hrdocs-knowledge-source`: `hrdocs` 検索インデックスを参照
* `web-knowledge-source`: Web IQ サービスのリモート MCP サーバーを参照。key ベースの認証と `web` ツールを使用

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    McpServerAutoOutputParsing,
    McpServerKnowledgeSource,
    McpServerKnowledgeSourceParameters,
    McpServerStoredHeadersAuthentication,
    McpServerStoredHeadersParameters,
    McpServerTool,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
WEB_KNOWLEDGE_SOURCE_NAME = "web-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [HR_KNOWLEDGE_SOURCE_NAME, HEALTH_KNOWLEDGE_SOURCE_NAME, WEB_KNOWLEDGE_SOURCE_NAME]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Zava HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Zava health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

web_knowledge_source = McpServerKnowledgeSource(
    name=WEB_KNOWLEDGE_SOURCE_NAME,
    description="Web IQ (live web search)",
    mcp_server_parameters=McpServerKnowledgeSourceParameters(
        server_url="https://api.microsoft.ai/v3/mcp",
        authentication=McpServerStoredHeadersAuthentication(
            stored_headers_parameters=McpServerStoredHeadersParameters(
                {"headers": {"x-apikey": WEB_IQ_KEY}}
            )
        ),
        tools=[McpServerTool(name="web", output_parsing=McpServerAutoOutputParsing())],
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=web_knowledge_source)
print("Knowledge sources created")


## Step 3: マルチソース + Web の knowledge base を作成する

knowledge base は、次の要素を束ねるオーケストレーション層です。

1. データソース（Step 2 で作成した knowledge source 群）
2. クエリの理解と回答生成のための LLM（Azure OpenAI）
3. クエリの処理方法とレスポンスのフォーマット設定

このノートブックでは `outputMode` に `answerSynthesis` を指定し、アタッチした LLM が元のユーザークエリにも答えられるようにします。あわせて `retrievalReasoningEffort` を `low` に設定し、クエリプランニングと knowledge source の選択に LLM を使う構成にしています。

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-web-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Zava knowledge base combining HR and health documents with access to web search.",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use the HR and health indexes for company policy questions. Use Web IQ web tool for context from public webpages.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")


## Step 4: knowledge base にクエリを投げる

片方は社内データ、もう片方は最新の外部ベンチマークが必要となる質問を投げてみましょう。

* _"What mental health coverage does Zava offer?"_ → `healthdocs` から返るはず
* _"What are industry benchmarks for mental health benefits at tech companies?"_ → Web IQ 経由で **Web** から返るはず

knowledge base は agentic retrieval を使って次を行います。

1. クエリを分析し、2 つの異なるトピックドメインがあることを認識する
2. 焦点を絞ったサブクエリに分解する
3. 社内寄りの質問は search index へ、ベンチマーク寄りの質問は Web へルーティングする
4. 3 つすべてのソースに対して並行に検索を実行する
5. semantic re-ranking を使って結果をマージする

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    KnowledgeSourceParams,
    SearchIndexKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

hrdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
    always_query_source=True,
)
healthdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
    always_query_source=True,
)
web_knowledge_source_params = KnowledgeSourceParams(
    knowledge_source_name=WEB_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
    kind="mcpServer",
)

question = (
    "A new hire wants to know if Zava's benefits are competitive. "
    "What mental health coverage does Zava offer according to our internal health plan docs? "
    "And search the web for current industry benchmarks for mental health benefits at tech companies."
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[hrdocs_knowledge_source_params, healthdocs_knowledge_source_params, web_knowledge_source_params],
    include_activity=True,
    max_runtime_in_seconds=120,
)

result = knowledge_base_client.retrieve(retrieval_request=retrieval_request)
display(Markdown(result.response[0].content[0].text))


### activity log を確認する

今回の activity log には、Web IQ への呼び出しに対応する `mcpServer` ステップと、各検索インデックスへのクエリごとの `searchIndex` ステップが表示されます。

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### references を確認する

Web IQ の knowledge source については、references の `type` が `mcpServer` になります。記事のタイトル、本文、URL は JSON 文字列として `sourceData.content` の中に格納されています。

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 Source Hunt

上に表示された references を見て、次を特定してみましょう。

1. **Zava のメンタルヘルス補償** に関する質問にはどの knowledge source が答えましたか？
2. **業界ベンチマーク** に関する質問にはどの knowledge source が答えましたか？

## ボーナス: Copilot CLI からクエリする

すべての knowledge base は MCP サーバーのエンドポイントを公開しており、VS Code や GitHub Copilot CLI など MCP 互換のクライアントから追加できます。
Copilot CLI から knowledge base にクエリを投げてみたい場合は、[Copilot CLI sidequest](copilot-cli-sidequest.md) のセットアップ手順に従い、下のコマンドを実行して最新の Zava knowledge base を MCP サーバーとして利用してください。

In [ ]:
print("copilot mcp remove zava-kb")
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
print(f"copilot mcp add zava-kb \"{mcp_url}\" --header \"api-key={AZURE_SEARCH_ADMIN_KEY}\"")
print(f"copilot -i \"{question}\"")

## ✅ ミッション達成

**構築したもの:**

* ✓ **Web IQ MCP Knowledge Source**: MCP プロトコル経由でライブ Web を knowledge base に接続する 3 つ目の knowledge source。社内ドキュメントと並べてリアルタイムの結果を取得できます。
* ✓ **3 ソースの Knowledge Base**: HR docs、health docs、ライブ Web にまたがる単一の KB。agentic ルーティングが各サブクエスチョンを自動で適切なソースへ振り分けます。
* ✓ **ハイブリッド取得**: Zava の補償に関する質問は社内の health plan ドキュメントが、業界ベンチマークに関する質問は Web のライブ引用が回答しました。

➡️ 次へ: [Part 3: Fabric IQ を追加する](part3-fabric-iq-to-kb.ipynb)。